In [1]:
import torch
from torch import nn

### Loss Function (Partial Cross-Entropy Loss)

In [2]:
class FocalLoss(nn.Module):
    def __init__(self, weights=None, num_classes=2, gamma=0.0, device="cpu"):
        super().__init__()

        self.device = device

        self.gamma = gamma
        self.num_classes = num_classes

        if weights is None:
            if num_classes > 2:
                classes = [1.0 for _ in range(num_classes)]
            else:
                classes = [1.0]

            weights = torch.tensor(classes).to(self.device)

        self.weights = weights
        self.nll_loss = nn.NLLLoss(weight=weights, reduction="none").to(device)

    def forward(self, input, target):
        input = (
            input.permute(0, *range(2, input.ndim), 1)
            .contiguous()
            .view(-1, self.num_classes)
            .to(self.device)
        )
        target = target.view(-1).to(self.device)

        probs = torch.softmax(input, dim=-1)
        nll_loss = self.nll_loss(torch.log_softmax(input, dim=-1), target)
        focal_term = (1 - probs[torch.arange(len(input)), target]) ** self.gamma

        return focal_term * nll_loss

In [3]:
class PartialCrossEntropyLoss(nn.Module):
    def __init__(
        self, num_classes, weights=None, gamma=0.0, reduce=False, device="cpu"
    ):
        super().__init__()

        self.reduce = reduce
        self.device = device

        self.focal_loss = FocalLoss(
            weights=weights, num_classes=num_classes, gamma=gamma, device=device
        )

    def forward(self, input, target, mask=None):
        focal_loss = self.focal_loss(input, target)

        if mask is None:
            mask = torch.ones_like(target)

        mask = mask.to(self.device)

        focal_loss = mask * focal_loss.contiguous().view(mask.shape)

        if self.reduce:
            focal_loss = focal_loss.sum()
            focal_loss /= mask.sum()

        return focal_loss

In [4]:
device = "cuda"

gamma = 0.0
reduction = "none"
reduce = reduction != "none"
num_classes = 3

In [5]:
reduce

False

In [6]:
mask_shape = (32, 1, 224, 224)

In [7]:
num_classes = 3

In [8]:
if num_classes > 2:
    classes = [1.0 for _ in range(num_classes)]
else:
    classes = [1]

weights = torch.tensor(classes)

In [9]:
ce_loss = nn.CrossEntropyLoss(reduction=reduction, weight=weights).to(device)
focal_loss = FocalLoss(
    gamma=0.0, num_classes=num_classes, weights=weights, device=device
)
pce_loss = PartialCrossEntropyLoss(
    gamma=gamma, num_classes=num_classes, weights=weights, reduce=reduce, device=device
)

In [10]:
y_ = torch.randn(10, num_classes)
y = torch.randint(low=0, high=num_classes, size=(10,))

In [11]:
predicted_mask = torch.randn(*(mask_shape[0], 3, mask_shape[2], mask_shape[-1]))
predicted_mask_ = (
    predicted_mask.permute(0, *range(2, predicted_mask.ndim), 1)
    .contiguous()
    .view(-1, 3)
)

ground_truth = torch.randint(low=0, high=num_classes, size=mask_shape)

In [12]:
loss_ce = ce_loss(predicted_mask_.to(device), ground_truth.flatten().to(device))

In [13]:
loss_f = focal_loss(predicted_mask, ground_truth)

In [14]:
loss_pce = pce_loss(predicted_mask, ground_truth)

In [15]:
loss_f.view(mask_shape).shape

torch.Size([32, 1, 224, 224])

In [16]:
torch.isclose(loss_ce, loss_f)

tensor([True, True, True,  ..., True, True, True], device='cuda:0')

In [17]:
loss_ce

tensor([4.1193, 0.4025, 0.2688,  ..., 0.2402, 1.4104, 1.6148], device='cuda:0')

In [18]:
loss_f

tensor([4.1193, 0.4025, 0.2688,  ..., 0.2402, 1.4104, 1.6148], device='cuda:0')

In [19]:
loss_pce.flatten()

tensor([4.1193, 0.4025, 0.2688,  ..., 0.2402, 1.4104, 1.6148], device='cuda:0')

In [20]:
loss_pce

tensor([[[[4.1193, 0.4025, 0.2688,  ..., 1.4749, 1.5099, 1.7128],
          [0.6294, 0.8034, 0.3858,  ..., 2.4650, 2.2425, 1.8751],
          [0.7472, 0.7847, 0.3867,  ..., 3.2685, 1.9129, 1.6515],
          ...,
          [1.4605, 2.7058, 2.6940,  ..., 1.2835, 1.4788, 0.3355],
          [1.5824, 0.4089, 1.4297,  ..., 0.6668, 0.9929, 1.0806],
          [0.9257, 0.9603, 3.0108,  ..., 0.2641, 2.9061, 1.0629]]],


        [[[1.9556, 1.2395, 1.1369,  ..., 1.0245, 1.3106, 1.7989],
          [1.6203, 2.4514, 1.8306,  ..., 0.2373, 1.6750, 1.2401],
          [2.2727, 1.0911, 1.1871,  ..., 3.0726, 1.6752, 2.3641],
          ...,
          [0.7974, 0.6446, 2.3918,  ..., 1.4267, 0.9994, 0.8050],
          [1.8442, 1.4423, 0.4396,  ..., 2.4068, 1.6073, 0.7710],
          [2.8537, 3.0649, 1.3100,  ..., 0.4940, 2.2879, 1.5696]]],


        [[[0.2401, 0.4933, 0.3797,  ..., 0.8832, 0.3719, 2.2383],
          [2.6047, 3.5685, 2.5936,  ..., 2.0434, 1.7105, 1.0588],
          [1.7371, 2.8809, 1.0130,  ..

### Mask Sampling

In [30]:
def sample_points_from_segmentation_mask(mask, num_samples=1000):
    B, C, H, W = mask.shape
    masked_H = torch.randint(low=0, high=H, size=(num_samples,))
    masked_W = torch.randint(low=0, high=W, size=(num_samples,))

    sampled_mask = torch.ones_like(mask) * -1
    sampled_mask[:, :, masked_H, masked_W] = mask[:, :, masked_H, masked_W]
    return sampled_mask, torch.where(sampled_mask == -1, 0, 1).to(torch.float32)

In [62]:
batch_predicted_mask = torch.randn(1, num_classes, 10, 10)
batch_ground_truth_mask = torch.randint(low=0, high=num_classes, size=(1, 1, 10, 10))

In [70]:
sampled_target_mask, filter_mask = sample_points_from_segmentation_mask(
    batch_ground_truth_mask, 10
)

In [71]:
sampled_target_mask, filter_mask

(tensor([[[[-1, -1, -1, -1, -1, -1,  2, -1, -1,  2],
           [-1, -1, -1, -1,  2, -1, -1, -1, -1, -1],
           [-1, -1, -1, -1, -1, -1, -1, -1, -1, -1],
           [-1, -1, -1, -1, -1, -1, -1, -1, -1, -1],
           [-1, -1, -1, -1, -1,  1, -1, -1, -1, -1],
           [-1, -1, -1, -1, -1, -1, -1, -1, -1, -1],
           [-1, -1, -1,  0,  0, -1, -1, -1,  0, -1],
           [-1, -1, -1, -1, -1, -1,  2, -1, -1,  1],
           [-1, -1, -1, -1, -1, -1, -1, -1, -1, -1],
           [-1, -1, -1, -1,  2, -1, -1, -1, -1, -1]]]]),
 tensor([[[[0., 0., 0., 0., 0., 0., 1., 0., 0., 1.],
           [0., 0., 0., 0., 1., 0., 0., 0., 0., 0.],
           [0., 0., 0., 0., 0., 0., 0., 0., 0., 0.],
           [0., 0., 0., 0., 0., 0., 0., 0., 0., 0.],
           [0., 0., 0., 0., 0., 1., 0., 0., 0., 0.],
           [0., 0., 0., 0., 0., 0., 0., 0., 0., 0.],
           [0., 0., 0., 1., 1., 0., 0., 0., 1., 0.],
           [0., 0., 0., 0., 0., 0., 1., 0., 0., 1.],
           [0., 0., 0., 0., 0., 0., 0., 0.

In [65]:
B.sum()

tensor(9.)

In [66]:
batch_ground_truth_mask

tensor([[[[1, 0, 2, 0, 0, 2, 2, 2, 2, 2],
          [0, 0, 1, 0, 2, 1, 0, 2, 0, 1],
          [1, 2, 2, 1, 0, 2, 2, 1, 2, 2],
          [1, 1, 0, 2, 2, 1, 0, 1, 2, 0],
          [2, 0, 1, 1, 0, 1, 1, 1, 0, 0],
          [1, 0, 1, 0, 1, 1, 1, 0, 2, 0],
          [2, 2, 0, 0, 0, 0, 2, 2, 0, 2],
          [1, 1, 2, 0, 0, 2, 2, 0, 1, 1],
          [1, 1, 0, 2, 2, 1, 2, 1, 1, 0],
          [1, 0, 1, 0, 2, 1, 1, 2, 2, 0]]]])

In [67]:
batch_predicted_mask

tensor([[[[-1.2584e+00,  4.0354e-01, -4.5285e-01,  4.2190e-01, -5.2903e-01,
           -2.7530e-01,  5.4818e-01,  6.3401e-01,  7.1309e-01,  3.8228e-01],
          [ 1.3324e-01,  1.5251e-02, -1.8286e+00, -6.2538e-01,  5.9200e-01,
           -2.5588e-01, -3.4989e-01, -1.7989e-01,  1.0916e+00, -7.8866e-03],
          [ 3.1038e-01,  1.2265e-01,  4.9292e-02,  2.3636e-01, -1.9553e-01,
           -2.1128e-01,  1.0184e-01, -1.4543e+00,  2.8822e-01, -6.8966e-01],
          [ 7.5352e-01,  8.7973e-02, -3.0704e+00, -3.2276e-01,  4.4237e-01,
           -3.8267e-01,  9.1272e-01, -5.5279e-01, -1.2404e+00,  7.5554e-01],
          [-1.5057e+00, -6.7380e-01, -2.2294e-02,  7.2991e-01, -3.4834e-01,
           -1.7628e+00, -2.0179e-01, -1.7939e+00,  3.9222e-01,  9.7355e-01],
          [ 9.5844e-01, -1.2068e+00, -5.0134e-01,  4.3068e-01,  8.6604e-01,
            1.2779e+00,  1.1981e-01, -1.3049e-01, -1.3769e-01,  5.8438e-01],
          [-6.0915e-01,  2.5035e-01, -1.1483e+00,  1.1528e+00,  6.2527e-02,
      

In [73]:
loss_pce = pce_loss(batch_predicted_mask, sampled_target_mask, mask=filter_mask)

../aten/src/ATen/native/cuda/Loss.cu:186: nll_loss_forward_no_reduce_cuda_kernel: block: [0,0,0], thread: [96,0,0] Assertion `cur_target >= 0 && cur_target < n_classes` failed.
../aten/src/ATen/native/cuda/Loss.cu:186: nll_loss_forward_no_reduce_cuda_kernel: block: [0,0,0], thread: [97,0,0] Assertion `cur_target >= 0 && cur_target < n_classes` failed.
../aten/src/ATen/native/cuda/Loss.cu:186: nll_loss_forward_no_reduce_cuda_kernel: block: [0,0,0], thread: [98,0,0] Assertion `cur_target >= 0 && cur_target < n_classes` failed.
../aten/src/ATen/native/cuda/Loss.cu:186: nll_loss_forward_no_reduce_cuda_kernel: block: [0,0,0], thread: [99,0,0] Assertion `cur_target >= 0 && cur_target < n_classes` failed.
../aten/src/ATen/native/cuda/Loss.cu:186: nll_loss_forward_no_reduce_cuda_kernel: block: [0,0,0], thread: [65,0,0] Assertion `cur_target >= 0 && cur_target < n_classes` failed.
../aten/src/ATen/native/cuda/Loss.cu:186: nll_loss_forward_no_reduce_cuda_kernel: block: [0,0,0], thread: [66,0,0] 

RuntimeError: CUDA error: device-side assert triggered
CUDA kernel errors might be asynchronously reported at some other API call, so the stacktrace below might be incorrect.
For debugging consider passing CUDA_LAUNCH_BLOCKING=1
Compile with `TORCH_USE_CUDA_DSA` to enable device-side assertions.


In [74]:
sampled_target_mask

tensor([[[[-1, -1, -1, -1, -1, -1,  2, -1, -1,  2],
          [-1, -1, -1, -1,  2, -1, -1, -1, -1, -1],
          [-1, -1, -1, -1, -1, -1, -1, -1, -1, -1],
          [-1, -1, -1, -1, -1, -1, -1, -1, -1, -1],
          [-1, -1, -1, -1, -1,  1, -1, -1, -1, -1],
          [-1, -1, -1, -1, -1, -1, -1, -1, -1, -1],
          [-1, -1, -1,  0,  0, -1, -1, -1,  0, -1],
          [-1, -1, -1, -1, -1, -1,  2, -1, -1,  1],
          [-1, -1, -1, -1, -1, -1, -1, -1, -1, -1],
          [-1, -1, -1, -1,  2, -1, -1, -1, -1, -1]]]])